In [34]:
import duckdb

In [35]:
con = duckdb.connect(database='dados_duckdb.db',read_only=False)

In [36]:
df = con.execute("SELECT * FROM bronze_z0019").fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-01-02 22:18:34.818417
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-01-02 22:18:34.818417
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-01-02 22:18:34.818417
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-01-02 22:27:24.083228
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-01-02 22:27:24.083228
5,10003,PREGO,BT10,100,60,z0019_2.csv,2026-01-02 22:27:24.083228


In [37]:
df = con.execute("""
                 SELECT * FROM (
                 SELECT *,
                        ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao desc) AS row  from bronze_z0019
                 where data_ingestao >= '2026-01-02'
                 ) WHERE ROW = 1
                 """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-01-02 22:18:34.818417,1
1,10003,PREGO,BT10,100,60,z0019_2.csv,2026-01-02 22:27:24.083228,1
2,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-01-02 22:18:34.818417,1
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-01-02 22:27:24.083228,1
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-01-02 22:27:24.083228,1


In [38]:
df_final = df.drop(columns=['nome_arquivo','data_ingestao','row'])
df_final = df_final.rename(columns={"NATBR":"ID"})
df_final = df_final.rename(columns={"MAKTX":"NM_PRODUTO"})
df_final = df_final.rename(columns={"WERKS":"ID_CATEGORIA"})
df_final = df_final.rename(columns={"MAINS":"ID_FORNECEDOR"})
df_final = df_final.rename(columns={"LABST":"VL_PRECO"})

df_final.head(10)


,ID,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10002,MARTELO,BT50,100,1500
1,10003,PREGO,BT10,100,60
2,10001,PARAFUSO,BT10,100,100
3,10004,SERRA,BT50,100,200
4,10005,MACHADO,BT50,100,100


In [39]:
df_final.head(10)

,ID,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10002,MARTELO,BT50,100,1500
1,10003,PREGO,BT10,100,60
2,10001,PARAFUSO,BT10,100,100
3,10004,SERRA,BT50,100,200
4,10005,MACHADO,BT50,100,100


In [40]:
df2 = df_final
df2 = df2.astype(
    {
        'ID':int,
        'NM_PRODUTO':str,
        'ID_CATEGORIA':str,
        'ID_FORNECEDOR':int,
        'VL_PRECO':float
    }
)

In [41]:
df2.dtypes

ID                 int64
NM_PRODUTO        object
ID_CATEGORIA      object
ID_FORNECEDOR      int64
VL_PRECO         float64
dtype: object

In [42]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos_doca(
            id BIGINT,
            NM_PRODUTO TEXT,
            ID_CATEGORIA TEXT,
            ID_FORNECEDOR BIGINT,
            VL_PRECO FLOAT
            )
""")

In [32]:
df2.head(10)

,ID,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10001,PARAFUSO,BT10,100,100.0
1,10004,SERRA,BT50,100,200.0
2,10003,PREGO,BT10,100,60.0
3,10005,MACHADO,BT50,100,100.0
4,10002,MARTELO,BT50,100,1500.0


In [43]:
df_resultado = con.execute("select * from produtos_doca").fetchdf()
df_resultado.head(10)

,id,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO


In [44]:
con.execute("insert into produtos_doca select * from df2")


In [45]:
df_resultado = con.execute("select * from produtos_doca").fetchdf()
df_resultado.head(10)

,id,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10002,MARTELO,BT50,100,1500.0
1,10003,PREGO,BT10,100,60.0
2,10001,PARAFUSO,BT10,100,100.0
3,10004,SERRA,BT50,100,200.0
4,10005,MACHADO,BT50,100,100.0


In [47]:
con.close()